In [12]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal


In [13]:
load_dotenv()
model = ChatOllama(model ='minimax-m3:cloud', temperature=0.5)

class Feedback(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description='Give the sentiment of the feedback')

prompt1 = ChatPromptTemplate([
    ('human','Classify the sentiment of the following feedback text into positive or negative \n {feedback} {format_Instruction}')
])
prompt2 =  ChatPromptTemplate([
    ('human','Write an appropiate response to this positive feedback\n {feedback}')
])

prompt3 = ChatPromptTemplate([
    ('human','Write an appropiate response to this negative feedback\n {feedback}')
])
parser = StrOutputParser()
parser1 = PydanticOutputParser(pydantic_object=Feedback)

In [15]:
classifier_chain = prompt1 | model | parser1

runnableBranch = RunnableBranch(
    (lambda x: x.sentiment == 'positive' , prompt2 | model | parser),
    (lambda x: x.sentiment == 'negative' , prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
) 

chain = classifier_chain | runnableBranch
result =  chain.invoke({'feedback':'This is terriable smartphone','format_Instruction': parser1.get_format_instructions()})
print(result)
 

# How to Respond to Negative Feedback

Since the specific feedback wasn't shared, here's a professional, empathetic template you can adapt to most situations:

---

## **Response Framework**

### 1. **Acknowledge & Thank**
> *"Thank you for taking the time to share your experience with us. We truly value your feedback and are sorry to hear that your experience didn't meet expectations."*

### 2. **Show Empathy**
> *"I understand how frustrating this must have been, and I want you to know that your concerns are completely valid."*

### 3. **Take Responsibility (Without Over-Apologizing)**
> *"While this isn't the level of service we strive to provide, I'm committed to making things right."*

### 4. **Offer a Solution**
> *"To resolve this, I'd like to [specific action: refund/replacement/follow-up/manager escalation]. Would this work for you?"*

### 5. **Invite Further Conversation**
> *"If there's anything else I can do, please don't hesitate to reach out directly at [contact info]."*
